In [1]:
########## PART A ##########

from google.colab import files
uploaded = files.upload()

Saving customer_feedback.json to customer_feedback.json


In [3]:
!pip install pyspark

In [4]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Practical 1") \
    .getOrCreate()
print("!!!session created!!!")

!!!session created!!!


In [6]:
########## LOADING JSON DATASET ##########
df = spark.read.json("customer_feedback.json")
print("Dataset loaded successfully!")
print("Total rows:", df.count())
print("Number of columns:", len(df.columns))
df.printSchema()

Dataset loaded successfully!
Total rows: 250000
Number of columns: 6
root
 |-- customer: struct (nullable = true)
 |    |-- age: long (nullable = true)
 |    |-- id: string (nullable = true)
 |    |-- location: string (nullable = true)
 |    |-- loyalty_years: long (nullable = true)
 |    |-- name: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- feedback: struct (nullable = true)
 |    |-- product_category: string (nullable = true)
 |    |-- rating: long (nullable = true)
 |    |-- sentiment: double (nullable = true)
 |    |-- store_location: string (nullable = true)
 |    |-- text: string (nullable = true)
 |-- feedback_id: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- timestamp: string (nullable = true)



In [8]:
#############################################
df.show(5, truncate=False)
print("Columns:", df.columns)
df.describe().show()

+---------------------------------------------------------------------------+-----------+----------------------------------------------------------------------------------+------------------------------------+----------+-------------------+
|customer                                                                   |device_type|feedback                                                                          |feedback_id                         |platform  |timestamp          |
+---------------------------------------------------------------------------+-----------+----------------------------------------------------------------------------------+------------------------------------+----------+-------------------+
|{43, 443e2fbe-b5cd-4a07-946f-f328f0d9e7e0, New York, NY, 2, Allison Hill}  |Web Browser|{Beauty, 3, 0.129, Houston, TX, Products are okay, nothing special.}              |cedb0461-a80f-4eb5-87a7-6e4a3c2d3782|Web       |2022-04-27T14:11:00|
|{32, 5994b8d0-3983-439e-a6f1-9b2628

In [9]:
########## CHECKING NULL VALUES ##########
from pyspark.sql.functions import col, sum, when
df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
]).show()
df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c.replace(".", "_"))
    for c in [
        "customer.id",
        "customer.age",
        "customer.location",
        "customer.loyalty_years",
        "feedback.text",
        "feedback.rating",
        "feedback.sentiment",
        "feedback.product_category",
        "feedback.store_location"
    ]
]).show()

+--------+-----------+--------+-----------+--------+---------+
|customer|device_type|feedback|feedback_id|platform|timestamp|
+--------+-----------+--------+-----------+--------+---------+
|       0|          0|       0|          0|       0|        0|
+--------+-----------+--------+-----------+--------+---------+

+-----------+------------+-----------------+----------------------+-------------+---------------+------------------+-------------------------+-----------------------+
|customer_id|customer_age|customer_location|customer_loyalty_years|feedback_text|feedback_rating|feedback_sentiment|feedback_product_category|feedback_store_location|
+-----------+------------+-----------------+----------------------+-------------+---------------+------------------+-------------------------+-----------------------+
|          0|           0|                0|                     0|            0|              0|                 0|                        0|                      0|
+-----------+---

In [10]:
##########DUPLICATE ##########
total_rows = df.count()
unique_ids = df.select("feedback_id").distinct().count()
duplicate_count = total_rows - unique_ids
print("Total Records:", total_rows)
print("Unique Feedback IDs:", unique_ids)
print("Duplicate Feedback IDs:", duplicate_count)

Total Records: 250000
Unique Feedback IDs: 250000
Duplicate Feedback IDs: 0


In [11]:
#########BLANK STRINGS ##########
from pyspark.sql.functions import col, trim
string_columns = [
    "feedback_id",
    "customer.id",
    "customer.name",
    "customer.location",
    "feedback.text",
    "feedback.product_category",
    "feedback.store_location",
    "platform",
    "device_type",
    "timestamp"
]
for c in string_columns:
    blank_count = df.filter(trim(col(c)) == "").count()
    print("Blank entries in", c, ":", blank_count)

Blank entries in feedback_id : 0
Blank entries in customer.id : 0
Blank entries in customer.name : 0
Blank entries in customer.location : 0
Blank entries in feedback.text : 0
Blank entries in feedback.product_category : 0
Blank entries in feedback.store_location : 0
Blank entries in platform : 0
Blank entries in device_type : 0
Blank entries in timestamp : 0


In [13]:
#######negative ages##########
df.filter(col("customer.age") < 0) \
    .select("feedback_id", "customer.id", "customer.age") \
    .show(10, truncate=False)

+------------------------------------+------------------------------------+---+
|feedback_id                         |id                                  |age|
+------------------------------------+------------------------------------+---+
|82cb2cf9-b038-462e-973e-18b76d309278|c7e5e644-f4e2-4d11-806a-dc5214c33ce9|-14|
|3a5fd3dd-52fb-4792-a274-26633329c3f8|412d3e4d-ebee-4dd7-9532-24e3ab6ecd12|-3 |
|d17e961e-5bc4-4fff-8e37-31a80b47752b|ea5d5589-0bda-4ca9-896f-a3c88f3221c8|-2 |
|8a3a5223-6e87-4f0e-9d50-1b48f76bd580|162b68cb-49e0-490b-ba6b-6f8a602a1176|-3 |
|5468833a-ea07-4fc2-a1f2-229df59735e5|fb48bfaf-a211-4533-9020-4bedc2d309be|-12|
|114c8e8f-b01b-4191-b2b5-faf8153cce95|9d1e494d-d154-4256-8db2-7457e34ca963|-5 |
|b951f5e9-3695-42b4-a27b-b9b16951ed38|a702388c-581c-4a14-8c92-707cf2c15b5e|-1 |
|7c194423-3ffb-4c17-9e18-2662b6365762|0267a0ac-679c-4ded-af48-47acaa293f2a|-5 |
|db606314-7581-4211-9bd9-b404fcad12c2|dabeee4d-d7ac-4519-ace8-e809596ad090|-1 |
|7cfc0bf8-91b9-4c7e-9491-8aba41777184|b3

In [16]:
####INVALID TIMESTAMPS ##########
from pyspark.sql import functions as F
# Convert timestamps using multiple possible formats
df_dates = df.withColumn(
    "parsed_timestamp",
    F.coalesce(
        F.try_to_timestamp("timestamp"),
        F.try_to_timestamp(
            "timestamp", F.lit("dd/MM/yyyy H:mm")
        ),
        F.try_to_timestamp(
            "timestamp", F.lit("MM/dd/yyyy H:mm")
        )
    )
)
# Count invalid timestamps
invalid_dates = df_dates.filter(
    F.col("parsed_timestamp").isNull()
).count()

print("Invalid Timestamps:", invalid_dates)
df_dates.filter(
    F.col("parsed_timestamp").isNull()
).select("timestamp").show(10, truncate=False)


Invalid Timestamps: 0
+---------+
|timestamp|
+---------+
+---------+



In [17]:
########## DATA QUALITY REPORT ##########
import pandas as pd
quality_report = pd.DataFrame({
    "Data Quality Issue": [
        "Null Values",
        "Duplicate Feedback IDs",
        "Blank Strings",
        "Negative Customer Ages",
        "Negative Loyalty Years",
        "Invalid Feedback Ratings",
        "Invalid Sentiment Scores",
        "Invalid Timestamps"
    ],
    "Count": [
        0,
        duplicate_count,
        0,
        negative_ages,
        negative_loyalty,
        invalid_ratings,
        invalid_sentiment,
        invalid_dates
    ]
})
print("CUSTOMER FEEDBACK DATA QUALITY REPORT")
display(quality_report)

CUSTOMER FEEDBACK DATA QUALITY REPORT


,Data Quality Issue,Count
0,Null Values,0
1,Duplicate Feedback IDs,0
2,Blank Strings,0
3,Negative Customer Ages,507
4,Negative Loyalty Years,0
5,Invalid Feedback Ratings,0
6,Invalid Sentiment Scores,0
7,Invalid Timestamps,0


In [19]:
########## DATA CLEANING ##########
from pyspark.sql.functions import col, when, lit
# Extract customer age into a separate column
valid_ages = df.select(
    col("customer.age").alias("age")
).filter(col("age") >= 0)
# Calculate median of valid customer ages
median_age = valid_ages.approxQuantile(
    "age", [0.5], 0.01
)[0]
print("Median Customer Age:", median_age)
# Replace negative ages with median age
df_cleaned = df.withColumn(
    "customer",
    col("customer").withField(
        "age",
        when(
            col("customer.age") < 0,
            lit(int(median_age))
        ).otherwise(col("customer.age"))
    )
)

print("Data cleaning completed!")

Median Customer Age: 41.0
Data cleaning completed!


In [20]:
########## VERIFY CLEANED DATA ##########
from pyspark.sql.functions import col
# Check remaining negative ages
remaining_negative_ages = df_cleaned.filter(
    col("customer.age") < 0
).count()
# Compare original and cleaned records
print("Original Records:", df.count())
print("Cleaned Records:", df_cleaned.count())
print("Remaining Negative Ages:", remaining_negative_ages)
# Display cleaned customer information
df_cleaned.select(
    "customer.id",
    "customer.age",
    "customer.loyalty_years"
).show(10, truncate=False)

Original Records: 250000
Cleaned Records: 250000
Remaining Negative Ages: 0
+------------------------------------+---+-------------+
|id                                  |age|loyalty_years|
+------------------------------------+---+-------------+
|443e2fbe-b5cd-4a07-946f-f328f0d9e7e0|43 |2            |
|5994b8d0-3983-439e-a6f1-9b2628a723fe|32 |9            |
|52e22034-0d5f-49cd-94ab-fde980a878ad|25 |0            |
|c64ca121-b4f1-4cbd-b29a-739d03131cc4|45 |1            |
|68a88cda-a73c-4284-b085-27cdb9232213|55 |1            |
|819455fd-aecc-4356-a0d2-67374f65d164|50 |3            |
|82873d10-c0cb-43a6-87a5-477687de6a11|48 |4            |
|4089062e-6558-4ba8-92ad-a5b467207944|58 |3            |
|de73e8f3-4d5f-4727-b310-6bc2ed92624d|42 |1            |
|34c34e29-2140-40e5-9066-fb4b203b42d5|40 |1            |
+------------------------------------+---+-------------+
only showing top 10 rows


In [22]:
# Save cleaned dataset as Parquet
df_cleaned.write.mode("overwrite") \
    .parquet("customerfeedback_cleaned.parquet")
print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


In [25]:
########## COMPRESS CLEANED PARQUET DATASET ##########
import shutil
shutil.make_archive(
    "customerfeedback_cleaned",
    "zip",
    ".",
    "customerfeedback_cleaned.parquet"
)
print("ZIP file created successfully!")

ZIP file created successfully!


In [26]:
from google.colab import files
files.download("customerfeedback_cleaned.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>